In [2]:
# Cell — force restart via magic command
%reset -f

In [3]:
# Cell 1 — Setup
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.data_split import load_features, time_based_split
from src.models.evaluate import evaluate_predictions, evaluate_by_group
from src.models.baseline_driver_rolling import predict_driver_rolling
from src.models.baseline_circuit_history import predict_circuit_history
from src.models.baseline_team_median import predict_team_median

df = load_features()
train, val, test = time_based_split(df)

Train: 1,311 rows (2021–2023)
Val:   458 rows (2024)
Test:  694 rows (2025 onward)


In [4]:
# Cell 2 — Run all three baselines on validation set
val = val.copy()
val["Pred_A_DriverRolling"] = predict_driver_rolling(val).fillna(val["DeltaToFastest_s"].median())
val["Pred_B_CircuitHistory"] = predict_circuit_history(val).fillna(val["DeltaToFastest_s"].median())
val["Pred_C_TeamMedian"] = predict_team_median(val).fillna(val["DeltaToFastest_s"].median())

results = []
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_A_DriverRolling"], "A - Driver Rolling"))
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_B_CircuitHistory"], "B - Circuit History"))
results.append(evaluate_predictions(val["DeltaToFastest_s"], val["Pred_C_TeamMedian"], "C - Team Median"))

results_df = pd.DataFrame(results)
results_df

A - Driver Rolling: MAE=0.835s | RMSE=2.301s | MedianAE=0.363s
B - Circuit History: MAE=1.190s | RMSE=3.263s | MedianAE=0.464s
C - Team Median: MAE=0.885s | RMSE=2.360s | MedianAE=0.370s


,Model,MAE,RMSE,MedianAE
0,A - Driver Rolling,0.8353,2.3011,0.3626
1,B - Circuit History,1.1904,3.2631,0.4640
2,C - Team Median,0.8846,2.3604,0.3703


In [5]:
# Cell 3 — Visual comparison
import plotly.express as px

fig = px.bar(
    results_df, x="Model", y="MAE",
    title="Baseline Model Comparison — Mean Absolute Error",
    labels={"MAE": "MAE (seconds)"},
    text="MAE"
)
fig.update_traces(textposition="outside")
fig.show()

In [6]:
# Cell 4 — Which baseline wins on which circuits?
best_baseline = "Pred_C_TeamMedian"  # update to whichever wins overall
by_circuit = evaluate_by_group(val, "DeltaToFastest_s", best_baseline, "EventName")
by_circuit.head(10)

,EventName,MAE
6,British Grand Prix,2.479907
19,Saudi Arabian Grand Prix,2.434374
5,Belgian Grand Prix,1.581283
10,Emilia Romagna Grand Prix,1.365744
1,Australian Grand Prix,1.365701
13,Japanese Grand Prix,0.988432
11,Hungarian Grand Prix,0.982927
7,Canadian Grand Prix,0.906568
4,Bahrain Grand Prix,0.807099
17,Monaco Grand Prix,0.748413


In [7]:
# Cell 5 — Save the best baseline's metrics for later comparison against XGBoost
best_metrics = results_df.loc[results_df["MAE"].idxmin()]
best_metrics.to_frame().T.to_csv(PROJECT_ROOT / "models/metrics/baseline_best.csv", index=False)
print(f"Best baseline: {best_metrics['Model']} — MAE {best_metrics['MAE']}s")

Best baseline: A - Driver Rolling — MAE 0.8353s


In [8]:
by_circuit_a = evaluate_by_group(val, "DeltaToFastest_s", "Pred_A_DriverRolling", "EventName")
by_circuit_a.head(10)

,EventName,MAE
6,British Grand Prix,2.601220
19,Saudi Arabian Grand Prix,2.385325
10,Emilia Romagna Grand Prix,1.583620
5,Belgian Grand Prix,1.137480
16,Miami Grand Prix,0.883910
12,Italian Grand Prix,0.812915
13,Japanese Grand Prix,0.785530
8,Chinese Grand Prix,0.728870
4,Bahrain Grand Prix,0.724610
9,Dutch Grand Prix,0.699221
